# 237 — clustering without the responsiveness gate

Every concat run so far keeps an electrode only if it is **high-activity in at least one
condition** (`require_high_activity=True` in `lf_concat.build_concat_dataset`). That is a
real scientific choice and it has never been tested. This notebook runs the identical
pipeline with the gate **off**, as a first-class feature set called `concat_hg_all`, so the
two can be compared like for like and both appear in the visualizer and the run report.

**The question.** Does dropping non-responsive electrodes make the clustering better or
worse?

**Why the obvious metric will lie.** Non-responsive electrodes are near-flat. Given a free
cluster they will form one tight, well-separated "nothing happened here" group, and
silhouette rewards exactly that. So an ungated run can score *better* while telling you
*less* — it would be spending one of its K clusters on the absence of a response. The last
cell tests precisely that: it recomputes the score with the flattest cluster removed. If
the gain disappears, the gate was doing its job.

**Nothing else changes.** Same input directory, same conditions, same HG band, same K
range, same `fit_and_save`, so the only difference between `concat_hg` and
`concat_hg_all` is which electrodes are in the matrix.
**Measured before writing this notebook**, so you know what to expect: ungated the
dataset is **2946 electrodes** against **1267** passing the gate, so lifting it adds
**1679 electrodes and more than doubles the matrix**. The added electrodes are flatter
but not flat: mean |HG| **0.392 dB** against **0.639** for the gated set, peak **1.58**
against **2.51**. That matters for the prediction above - they are not a trivial block of
zeros that would obviously bunch into one cluster, so the test is worth running rather
than reasoning about.

One discrepancy to keep an eye on: **1267** electrodes pass the gate here against the
**1266** in the published `concat_hg` run. One electrode, almost certainly recovered by
the dash fix in `lf_dataset._is_non_neural` after that run was fitted. Harmless for the
comparison, but it means the gated and ungated runs are not on exactly the same base.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions import lf_concat as CC
from functions import lf_cluster_run as R

SCRIPT_NAME = '237_ungated_clustering.ipynb'

In [ ]:
# ── data input ───────────────────────────────
INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

# ── sample filter ────────────────────────────
CONDITIONS            = ('audio', 'picture', 'reading')
REQUIRE_HIGH_ACTIVITY = False    # <<< THE ONLY DIFFERENCE FROM 233
FEATURE_SET           = 'concat_hg_all'
FSET_LABEL            = 'Concatenated HG, ungated [a|p|r]'

# ── representation ───────────────────────────  (identical to 233)
HG_BAND = (70.0, 150.0)
FMAX    = 500.0

# ── clustering ───────────────────────────────  (identical to 233)
K_RANGE      = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
HC_METHOD    = 'ward'
HC_METRIC    = 'euclidean'
RANDOM_STATE = cfg.RANDOM_STATE

print('conditions   :', CONDITIONS, '| gate high-activity>=1:', REQUIRE_HIGH_ACTIVITY)
print('feature set  :', FEATURE_SET)
print('K_RANGE      :', K_RANGE)

## 1. Build the ungated dataset

`build_concat_dataset` still requires all three conditions to be present — that is a
completeness requirement, not a responsiveness one, and it stays on. Only the
high-activity gate is lifted.

In [ ]:
df_all, X_concat_all = CC.build_concat_dataset(
    INPUT_DIR, conditions=CONDITIONS, require_high_activity=REQUIRE_HIGH_ACTIVITY)

gate_pass = df_all['n_high_activity'].to_numpy() > 0
print(f'\nungated dataset : {len(df_all)} electrodes · {df_all["patient_id"].nunique()} patients')
print(f'  would pass the gate : {int(gate_pass.sum())}')
print(f'  added by ungating   : {int((~gate_pass).sum())} '
      f'({100*(~gate_pass).mean():.0f}% of the ungated set)')
df_all.head()

## 2. What does ungating actually add?

If the added electrodes are flat, expect them to form their own cluster later. This cell
says how flat they are *before* any clustering, so the later result is not a surprise.

In [ ]:
X_hg_all = CC.concat_hg_features(X_concat_all, hg_band=HG_BAND, fmax=FMAX)
print('concat_hg_all:', X_hg_all.shape)

amp  = np.abs(X_hg_all).mean(axis=1)      # mean |HG| across the whole concatenated trial
peak = np.abs(X_hg_all).max(axis=1)       # strongest excursion anywhere in it

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, v, name in ((axes[0], amp, 'mean |HG| (dB)'), (axes[1], peak, 'peak |HG| (dB)')):
    bins = np.linspace(0, np.percentile(v, 99.5), 60)
    ax.hist(v[gate_pass],  bins=bins, alpha=.75, label=f'passes gate (n={int(gate_pass.sum())})',
            color='#41ab5d')
    ax.hist(v[~gate_pass], bins=bins, alpha=.75, label=f'added by ungating (n={int((~gate_pass).sum())})',
            color='#c1121f')
    ax.set_xlabel(name); ax.set_ylabel('# electrodes'); ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('What the responsiveness gate removes', x=.01, ha='left')
plt.tight_layout(); plt.show()

print(f'mean |HG| : gated {amp[gate_pass].mean():.3f} dB   vs added {amp[~gate_pass].mean():.3f} dB')
print(f'peak |HG| : gated {peak[gate_pass].mean():.2f} dB    vs added {peak[~gate_pass].mean():.2f} dB')

## 3. Fit k-means and Ward

Same call as 233, only `feature_set` differs. `fit_and_save` writes the run directory,
`X_train.npy`, `labels.csv`, `feature_schema.json` and registers it in `index.json`, which
is what every downstream tool reads.

In [ ]:
m = R.fit_and_save(
    X_hg_all, df_keep=df_all, method='kmeans', feature_set=FEATURE_SET,
    params={'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means', feature_set_label=FSET_LABEL,
    feature_names=CC.concat_feature_names(FEATURE_SET), notebook=SCRIPT_NAME)
print('best K (kmeans/%s):' % FEATURE_SET, m['summary']['best_k'])

m = R.fit_and_save(
    X_hg_all, df_keep=df_all, method='hierarchical', feature_set=FEATURE_SET,
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': K_RANGE},
    method_label='Hierarchical', feature_set_label=FSET_LABEL,
    feature_names=CC.concat_feature_names(FEATURE_SET), notebook=SCRIPT_NAME)
print('best K (hierarchical/%s):' % FEATURE_SET, m['summary']['best_k'])

## 4. Convex NMF

cNMF is not fitted in a notebook anywhere in this project — `run_decomposition.py` fits it
and `publish_decomposition.py` writes it out as a run. `concat_hg_all` is registered in
both, and its source resolves to the **newest k-means run for this feature set**, i.e. the
one the cell above just wrote. Run these two from `02_FBM_Clustering/`:

```
python run_decomposition.py --feature-set concat_hg_all
python publish_decomposition.py --feature-set concat_hg_all
```

The next cell will do it for you.

In [ ]:
import subprocess
for cmd in (['run_decomposition.py', '--feature-set', FEATURE_SET],
            ['publish_decomposition.py', '--feature-set', FEATURE_SET]):
    print('\n$ python', ' '.join(cmd))
    p = subprocess.run([sys.executable] + cmd, cwd=str(Path.cwd()),
                       capture_output=True, text=True)
    print(p.stdout[-2500:])
    if p.returncode:
        print('!! FAILED\n', p.stderr[-2000:])
        break

## 5. Make it visible — visualizer and full report

These are the same steps every other track goes through. Run 252 first (it needs the runs
to exist), then the rest from `02_FBM_Clustering/`:

```
# 252_clustering_recon.ipynb        <- glassbrain recon for the new runs
python make_missing_centroids.py                 # centroid chips
python make_centroid_rasters.py                  # per-cluster rasters
python make_decomposition_drivers.py             # D1/D2, cnmf only
python render_cnmf_glassbrains.py --published --skip-existing --which cluster --scale 1
python pack_spin_frames.py
python make_cluster_cards.py --run outputs/clustering/kmeans/concat_hg_all/runs/<id>
python make_coverage_bundle.py                   # <- the visualizer reads this
```

`make_coverage_bundle.py` already lists the three `concat_hg_all` tracks under their own
cohort, so the new runs appear in the run switcher and their report is complete. Then
**commit and push** — the report fetches its figures from the repo, so anything untracked
shows as *not published for this run*.

## 6. The test the notebook exists for

Silhouette in the space each method fits in (k-means and Ward on raw dB), against a
one-blob null built the same way — the same procedure as FIG C.7. Then the part that
matters: **recompute with the flattest cluster dropped**. If the ungated advantage
disappears, the extra separation was the gate's absence, not new signal.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from functions import lf_runs as LR

K_TEST = 7

def null_silhouette(X, k=K_TEST, reps=8):
    Xc = X - X.mean(0); n = len(X)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    out = []
    for i in range(reps):
        Z = np.random.default_rng(500 + i).standard_normal((n, len(S)))
        Y = (Z * (S / np.sqrt(n - 1))) @ Vt
        out.append(silhouette_score(Y, KMeans(k, n_init=10, random_state=i).fit_predict(Y)))
    return float(np.mean(out)), float(np.std(out))

def report(name, X, lab, amp_vec):
    nm, ns = null_silhouette(X)
    s = silhouette_score(X, lab)
    print(f'{name:<28} silhouette {s:+.4f}   null {nm:.4f}+/-{ns:.4f}   z {(s-nm)/max(ns,1e-9):+.1f}')
    # which cluster is the flat one, and what happens without it
    means = {c: amp_vec[lab == c].mean() for c in np.unique(lab)}
    flat = min(means, key=means.get)
    keep = lab != flat
    s2 = silhouette_score(X[keep], lab[keep])
    print(f'{"":<28}   flattest cluster c{flat}: n={int((lab==flat).sum())}, '
          f'mean |HG| {means[flat]:.3f} dB (run mean {amp_vec.mean():.3f})')
    print(f'{"":<28}   silhouette WITHOUT it {s2:+.4f}   '
          f'({"gain survives" if s2 >= s - 0.005 else "gain came from that cluster"})')
    return s, s2

run_all = LR.newest_run('kmeans', FEATURE_SET)
lab_all = pd.read_csv(run_all / 'cluster_labels_by_k.csv')[f'k_{K_TEST}'].to_numpy()
X_all   = np.load(run_all / 'X_train.npy')
report('kmeans / concat_hg_all', X_all, lab_all, np.abs(X_all).mean(1))

print()
run_gt = LR.newest_run('kmeans', 'concat_hg')
lab_gt = pd.read_csv(run_gt / 'cluster_labels_by_k.csv')[f'k_{K_TEST}'].to_numpy()
X_gt   = np.load(run_gt / 'X_train.npy')
report('kmeans / concat_hg (gated)', X_gt, lab_gt, np.abs(X_gt).mean(1))